# Leitura de gabarito com OpenCV
Detecta as bolhas de um cartão-resposta, descobre qual está pintada em cada questão e devolve as respostas.
`?` = nenhuma marcada · `*` = duas ou mais marcadas.

In [ ]:
from google.colab import files
enviados = files.upload()

In [ ]:
import cv2, numpy as np
from matplotlib import pyplot as plt

ALT        = "ABCDE"
N_ALT      = 5      # alternativas por questão
BOLHA_MIN  = 25     # menor lado de uma bolha, em pixels
MIOLO      = 0.60   # mede só o centro da bolha (ignora o círculo e a letra impressos)
PREENCHIDA = 0.70   # % do miolo pintado para contar como marcada

def _cortar(itens, chave, limite):
    """Quebra uma sequência ordenada onde o vão entre um item e o próximo passa do limite."""
    grupos, atual = [], [itens[0]]
    for a, b in zip(itens, itens[1:]):
        if chave(b) - chave(a) > limite:
            grupos.append(atual); atual = []
        atual.append(b)
    return grupos + [atual]

def ler_gabarito(caminho):
    img = cv2.imread(caminho)
    cinza = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, masc = cv2.threshold(cinza, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 1) bolhas = contornos externos redondos e grandes o suficiente
    bolhas = []
    for c in cv2.findContours(masc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x, y, w, h = cv2.boundingRect(c)
        if w >= BOLHA_MIN and h >= BOLHA_MIN and 0.7 <= w / h <= 1.4:
            bolhas.append((x, y, w, h, c))

    # 2) linhas: bolhas na mesma altura
    bolhas.sort(key=lambda b: b[1])
    altura = np.median([b[3] for b in bolhas])
    linhas = [sorted(l, key=lambda b: b[0]) for l in _cortar(bolhas, lambda b: b[1], altura * 0.6)]

    # 3) bandas: fileiras de blocos empilhadas (a folha real tem duas)
    vaos_v = [b[0][1] - a[0][1] for a, b in zip(linhas, linhas[1:])]
    bandas = _cortar(linhas, lambda l: l[0][1], np.median(vaos_v) * 1.5) if vaos_v else [linhas]

    # 4) preenchimento do miolo de uma bolha
    def preenchimento(c):
        (cx, cy), r = cv2.minEnclosingCircle(c)
        m = np.zeros(masc.shape, np.uint8)
        cv2.circle(m, (int(cx), int(cy)), max(int(r * MIOLO), 1), 255, -1)
        return cv2.countNonZero(cv2.bitwise_and(masc, m)) / max(cv2.countNonZero(m), 1)

    respostas, saida, n = {}, img.copy(), 0
    for banda in bandas:
        blocos = 0
        for i, linha in enumerate(banda):
            # 5) blocos: dentro da linha, o vão entre blocos é bem maior que entre alternativas
            vaos_h = [b[0] - a[0] for a, b in zip(linha, linha[1:])]
            celulas = _cortar(linha, lambda b: b[0], np.median(vaos_h) * 1.8) if vaos_h else [linha]
            celulas = [c for c in celulas if len(c) == N_ALT]
            blocos = max(blocos, len(celulas))

            for j, celula in enumerate(celulas):
                q = n + j * len(banda) + i + 1      # blocos numerados da esquerda para a direita
                cheio = [preenchimento(c) for *_, c in celula]
                marcadas = [k for k, p in enumerate(cheio) if p >= PREENCHIDA]
                respostas[q] = ("?" if not marcadas else
                                "*" if len(marcadas) > 1 else ALT[marcadas[0]])

                cor = (0, 180, 0) if respostas[q] in ALT else (0, 0, 255)
                for k in (marcadas or range(len(celula))):
                    x, y, w, h, _ = celula[k]
                    cv2.rectangle(saida, (x, y), (x + w, y + h), cor, 2)
                x, y, w, h, _ = celula[-1]
                cv2.putText(saida, f"{q}{respostas[q]}", (x + w + 3, y + h - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, cor, 1)
        n += blocos * len(banda)

    return [respostas[q] for q in sorted(respostas)], masc, saida

In [ ]:
for arquivo in enviados:
    respostas, mascara, saida = ler_gabarito(arquivo)
    print(f"\n{arquivo} — {len(respostas)} questões")
    print(" ".join(f"{i}{r}" for i, r in enumerate(respostas, 1)))

    fig, ax = plt.subplots(1, 2, figsize=(16, 7))
    ax[0].imshow(mascara, cmap="gray");                      ax[0].set_title("o que o OpenCV enxerga")
    ax[1].imshow(cv2.cvtColor(saida, cv2.COLOR_BGR2RGB));    ax[1].set_title("leitura")
    for a in ax: a.axis("off")
    plt.show()